# Axiom Data Acceptance Console

Read-only public API console. Edit CONFIG and rerun to inspect an exact Snapshot.
The supplied configuration is a bounded historical fixture, **not a V1 baseline**.
No cell collects, builds, publishes or changes Data artifacts. Dates describe query
scope; data extents are not full-scope certification. Historical bootstrap remains
best-effort unless revision-specific evidence supports stronger qualification.


In [ ]:
from pathlib import Path
import hashlib
import pandas as pd
from axiom_data import SnapshotReader, FactView
from axiom_data.operations import resolve_snapshot, inspect_snapshot, compare_event_projection
CONFIG = {'data_root': '/home/liuming/workspace/axiom/data/forensic/pr7-dm2-20260909-r1/admission-r5/source', 'snapshot_id': 'snapshot-fa3d8b80c729edd2e139a49c82b3dea868bb5639998ba7cc84f540c0623fe786', 'views': {'financial_view_id': 'pr6-fact-1abe9f256a3c7f501b4596974f4b763b30a2885ee435de0a887b83f0d8c0ed4e', 'adjusted_view_id': 'adjusted-price-5722f89cb9640a0caaeb1c653d6bda330e8ffecd7e5f0987a895634051c2c47b', 'event_view_id': 'pr7-fact-bbec4919922368305ee96afa507304b78a85d6bdd0d05f76883f917f357cebb5'}, 'symbol': '688981.SH', 'start': '2025-06-10', 'end': '2025-06-13', 'cutoff': '2025-06-13T23:59:59+08:00', 'pit_policy': 'best_effort_vendor_v1', 'universe': '000906.SH', 'industry_system': 'tushare_bak_basic'}
SNAPSHOT_ID = resolve_snapshot(CONFIG["data_root"], CONFIG["snapshot_id"])
import axiom_data.operations as service
CODE_VERSION = {"operations_sha256": hashlib.sha256(Path(service.__file__).read_bytes()).hexdigest()}
display({"snapshot_id": SNAPSHOT_ID, "view_refs": CONFIG["views"], "code_version": CODE_VERSION, "config": CONFIG})
reader = SnapshotReader(CONFIG["data_root"], SNAPSHOT_ID)
facts = FactView(CONFIG["data_root"], SNAPSHOT_ID, adjusted_price_view_id=CONFIG["views"]["adjusted_view_id"], financial_fact_view_id=CONFIG["views"]["financial_view_id"])


## Snapshot and quality

Domain refs, schema checks, actual data extents and qualification. Full-scope admission
is shown separately and must be explicitly assessed.


In [ ]:
inventory = inspect_snapshot(CONFIG["data_root"], SNAPSHOT_ID)
display(pd.DataFrame(inventory["domains"]).T)


## Security / market

Unadjusted and explicitly anchored prices; status, limits and corporate actions.


In [ ]:
SYMBOL = CONFIG["symbol"]
START, END = CONFIG["start"], CONFIG["end"]
display(pd.DataFrame(reader.security_master([SYMBOL])))
display(pd.DataFrame(reader.market_daily([SYMBOL], START, END)))
for domain in ("security_status", "price_limits", "corporate_actions"):
    display({"domain": domain})
    display(pd.DataFrame(reader.facts(domain, symbols=[SYMBOL], start_session=START, end_session=END)))
adjusted = facts.read("adjusted_price", symbols=[SYMBOL], start_session=START, end_session=END, price_basis="anchor_adjusted", pit_policy="research_non_pit")
display({key: value for key, value in adjusted.items() if key != "rows"})
display(pd.DataFrame(adjusted["rows"]))


## PIT / financial

Change CUTOFF and rerun. Selectors and single-quarter/TTM calculations run only in
public Reader methods. Component refs and missing reasons remain visible.


In [ ]:
CUTOFF = CONFIG["cutoff"]
POLICY = CONFIG["pit_policy"]
query = {"knowledge_cutoff": CUTOFF, "pit_policy": POLICY}
for group, domain in ((CONFIG["universe"], "universe_membership"), (CONFIG["industry_system"], "industry_membership")):
    display({"group": group, "cutoff": CUTOFF})
    display(pd.DataFrame(reader.members(group, END, domain=domain, symbols=[SYMBOL], **query)))
display(pd.DataFrame(reader.as_of("financial_events", symbols=[SYMBOL], **query)))
display(pd.DataFrame(reader.financial_derived(symbols=[SYMBOL], **query)))


## Other facts and revisions

The cutoff is applied to holder/Top10 and forecast revisions. Raw daily margin and
moneyflow are also shown through the PIT Reader. No Notebook projection logic.


In [ ]:
for domain in ("holder_count_events", "top_holders_reports", "margin_daily", "moneyflow_daily", "forecast_observations"):
    display({"domain": domain, "cutoff": CUTOFF})
    display(pd.DataFrame(reader.as_of(domain, symbols=[SYMBOL], **query)))


## Direct / Qlib equivalence

The public acceptance service checks keys, missingness and numeric tolerance.


In [ ]:
display(compare_event_projection(CONFIG["data_root"], SNAPSHOT_ID, CONFIG["views"]["event_view_id"], symbols=[SYMBOL], fields=["holder.number", "holder.top10_ratio", "margin.balance"], start_session=START, end_session=END))


## Reconciliation and limitations

Use an explicitly Snapshot-bound validation/reconciliation report supplied by the
release decision. This console does not infer agreement with Qsys from row counts
or recompute source mappings. A V1 full-scope report will replace the bounded
fixture configuration only after actual acceptance.
